In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

RAW_CSV = "/home/kulikoval/mmkorshunova/Optimal-Stop-Problem/output/metrics_draft/my-fi.csv"
OUT_DIR = "/home/kulikoval/mmkorshunova/Optimal-Stop-Problem/output/summary_diploma"
os.makedirs(OUT_DIR, exist_ok=True)

Z_95 = 1.96  # для 95% CI по среднему
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)


In [ ]:
df = pd.read_csv(RAW_CSV)

# Приводим числовые колонки
num_cols = [
    "price", "duration", "time_path_gen", "comp_time",
    "delta", "gamma", "theta", "rho", "vega",
    "nb_paths", "nb_dates", "spot", "strike", "maturity",
    "volatility", "drift", "dividend", "nb_stocks"
]

for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Фильтр: только строки, где есть price (успешные прогоны)
df = df.dropna(subset=["price"]).copy()

# В идеале должно быть 3 algo * 4 nb_dates * 20 runs = 240 строк
print("Rows:", len(df))
display(df.head(5))
print("\nAlgos:", sorted(df["algo"].unique()))
print("nb_dates:", sorted(df["nb_dates"].dropna().unique()))


In [ ]:
# Проверим, что параметры едины (кроме nb_dates)
key_cols = ["model", "payoff", "spot", "strike", "maturity", "volatility", "drift", "dividend", "nb_paths", "nb_stocks"]
profile = df[key_cols].drop_duplicates()
print("Unique scenario profiles (should be small):", len(profile))
display(profile)

# Проверим, есть ли delta
delta_missing = df["delta"].isna().mean() if "delta" in df.columns else 1.0
print("Share of missing delta:", delta_missing)

# Сколько прогонов на (algo, nb_dates)
counts = df.groupby(["algo", "nb_dates"]).size().unstack(fill_value=0)
display(counts)


In [ ]:
group_cols = ["algo", "nb_dates"]

def summarize_group(g: pd.DataFrame) -> pd.Series:
    n = len(g)

    price_mean = g["price"].mean()
    price_std = g["price"].std(ddof=1) if n > 1 else 0.0
    price_se = price_std / np.sqrt(n) if n > 0 else np.nan
    price_ci_half = Z_95 * price_se

    duration_mean = g["duration"].mean()
    duration_std = g["duration"].std(ddof=1) if n > 1 else 0.0

    time_path_gen_mean = g["time_path_gen"].mean() if "time_path_gen" in g.columns else np.nan
    comp_time_mean = g["comp_time"].mean() if "comp_time" in g.columns else np.nan

    delta_mean = g["delta"].mean() if "delta" in g.columns else np.nan
    delta_std = g["delta"].std(ddof=1) if ("delta" in g.columns and n > 1) else np.nan

    return pd.Series({
        "n_runs": n,
        "price_mean": price_mean,
        "price_ci95_half": price_ci_half,
        "price_ci95_low": price_mean - price_ci_half,
        "price_ci95_high": price_mean + price_ci_half,
        "duration_mean": duration_mean,
        "duration_std": duration_std,
        "time_path_gen_mean": time_path_gen_mean,
        "comp_time_mean": comp_time_mean,
        "delta_mean": delta_mean,
        "delta_std": delta_std,
    })

pricing_summary = df.groupby(group_cols, dropna=False).apply(summarize_group).reset_index()
pricing_summary = pricing_summary.sort_values(["nb_dates", "algo"]).reset_index(drop=True)

display(pricing_summary)

out_path = os.path.join(OUT_DIR, "pricing_summary_by_nb_dates.csv")
pricing_summary.to_csv(out_path, index=False)
print("Saved:", out_path)


In [ ]:
group_cols = ["algo", "nb_dates"]

def summarize_group(g: pd.DataFrame) -> pd.Series:
    n = len(g)

    price_mean = g["price"].mean()
    price_std = g["price"].std(ddof=1) if n > 1 else 0.0
    price_se = price_std / np.sqrt(n) if n > 0 else np.nan
    price_ci_half = Z_95 * price_se

    duration_mean = g["duration"].mean()
    duration_std = g["duration"].std(ddof=1) if n > 1 else 0.0

    time_path_gen_mean = g["time_path_gen"].mean() if "time_path_gen" in g.columns else np.nan
    comp_time_mean = g["comp_time"].mean() if "comp_time" in g.columns else np.nan

    delta_mean = g["delta"].mean() if "delta" in g.columns else np.nan
    delta_std = g["delta"].std(ddof=1) if ("delta" in g.columns and n > 1) else np.nan

    return pd.Series({
        "n_runs": n,
        "price_mean": price_mean,
        "price_ci95_half": price_ci_half,
        "price_ci95_low": price_mean - price_ci_half,
        "price_ci95_high": price_mean + price_ci_half,
        "duration_mean": duration_mean,
        "duration_std": duration_std,
        "time_path_gen_mean": time_path_gen_mean,
        "comp_time_mean": comp_time_mean,
        "delta_mean": delta_mean,
        "delta_std": delta_std,
    })

pricing_summary = df.groupby(group_cols, dropna=False).apply(summarize_group).reset_index()
pricing_summary = pricing_summary.sort_values(["nb_dates", "algo"]).reset_index(drop=True)

display(pricing_summary)

out_path = os.path.join(OUT_DIR, "pricing_summary_by_nb_dates.csv")
pricing_summary.to_csv(out_path, index=False)
print("Saved:", out_path)


In [ ]:
# Price vs nb_dates with CI half-width
plt.figure()
for algo in sorted(pricing_summary["algo"].unique()):
    sub = pricing_summary[pricing_summary["algo"] == algo].sort_values("nb_dates")
    plt.errorbar(sub["nb_dates"], sub["price_mean"], yerr=sub["price_ci95_half"], marker="o", linestyle="-", label=algo)

plt.xlabel("nb_dates (n)")
plt.ylabel("Price mean ± 95% CI")
plt.title("Pricing sensitivity: price vs nb_dates")
plt.legend()
plt.grid(True)
plt.show()

# Duration vs nb_dates
plt.figure()
for algo in sorted(pricing_summary["algo"].unique()):
    sub = pricing_summary[pricing_summary["algo"] == algo].sort_values("nb_dates")
    plt.plot(sub["nb_dates"], sub["duration_mean"], marker="o", linestyle="-", label=algo)

plt.xlabel("nb_dates (n)")
plt.ylabel("Mean duration (s)")
plt.title("Pricing sensitivity: runtime vs nb_dates")
plt.legend()
plt.grid(True)
plt.show()
